# Scam Checker
Type any text message and see the chance it's a scam, plus the action a mobile carrier would take.

This notebook uses the final model from Final_Report.ipynb (Model C: balanced logistic regression on TF-IDF word features plus 10 text signals). Run the final report first so the saved model models/scam_model.joblib exists. No retraining happens here, so it runs in a second.

# How it works

1. Load the saved model. The file holds three pieces, all fit on the training data only: the TF-IDF vocabulary, the scaler for the text signals, and the logistic regression model.
2. Turn the message into features the same way the model was trained: TF-IDF word scores, plus the 10 text signals from src/scamfeatures.py (digits, phone number, short code, link, currency symbol, capital letters, exclamation points, call to action, word count and sentiment).
3. Score it. The model returns the probability the message is a scam, and the carrier action follows from the cutoffs in the final report.

We block only at 95% or higher because scams are rare in real traffic: at the 50% cutoff, about half of flagged texts could be real messages if only 1% of texts are scams.

In [1]:
# Load the trained model and the shared feature function
import sys, joblib
from scipy.sparse import hstack, csr_matrix
sys.path.append("..")
from src.scamfeatures import signal_features

saved = joblib.load("../models/scam_model.joblib")
tfidf, scaler, model = saved["tfidf"], saved["scaler"], saved["model"]

def scam_test(message):
    words = tfidf.transform([message])
    signals = csr_matrix(scaler.transform(signal_features([message])))
    prob = model.predict_proba(hstack([words, signals]).tocsr())[0, 1]
    if prob >= 0.95:
        verdict = "🚫 Block"
    elif prob >= 0.5:
        verdict = "⚠️ Warn"
    else:
        verdict = "✅ Likely safe"
    print(f'Message: "{message}"')
    print(f"{verdict}: {prob * 100:.1f}% chance this is a scam")

# Examples

Four messages that show each outcome. The package scam is warned rather than blocked, and the real verification code also gets a warning because its digits push the score up. That is why the final report recommends an allowlist for verified senders such as banks and two factor login codes.

In [2]:
# One message for each outcome
for msg in ["Hey, are we still meeting for lunch at noon?",
            "URGENT! You have won a £1000 prize. Call 09061701461 now to claim. T&C apply",
            "Your package could not be delivered. Update your address at usps-redelivery.com",
            "Your verification code is 482913. Do not share it with anyone."]:
    scam_test(msg)

Message: "Hey, are we still meeting for lunch at noon?"
✅ Likely safe: 1.2% chance this is a scam
Message: "URGENT! You have won a £1000 prize. Call 09061701461 now to claim. T&C apply"
🚫 Block: 100.0% chance this is a scam
Message: "Your package could not be delivered. Update your address at usps-redelivery.com"
⚠️ Warn: 83.8% chance this is a scam
Message: "Your verification code is 482913. Do not share it with anyone."
⚠️ Warn: 62.7% chance this is a scam


# Try your own message

Run the cell below, type a message in the box that appears, and press Enter. The model was trained on older UK and Indian texts, so it may be less sure about modern U.S. scams.

In [3]:
# Run this cell, type a message in the box at the top, and press Enter
message = input("Enter a text message: ")
scam_test(message)

Message: "Your package could not be delivered. Update your address at usps-redelivery.com"
⚠️ Warn: 83.8% chance this is a scam
